In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def load_pfm(path):
    with open(path, "rb") as f:
        header = f.readline().decode("ascii").strip()
        if header not in ("PF", "Pf"):
            raise ValueError("Not a PFM file.")
        w, h = map(int, f.readline().decode("ascii").split())
        scale = float(f.readline().decode("ascii").strip())
        endian = "<" if scale < 0 else ">"
        channels = 3 if header == "PF" else 1
        data = np.fromfile(f, endian + "f")

    data = data.reshape((h, w, channels))
    
    return data[..., 0] if channels == 1 else data


In [ ]:
mask = cv2.imread(
    "/mnt/c/Users/shh/Documents/ShunHsiangHsu/DefectSynthetic/rhino_modeling/rendered/runs/cube/20260318_020454/mask/view_003.png",
)
trg_colors = [(0, 165, 255), (0, 128, 0), (0, 255, 255)]
trg_mask = np.zeros(mask.shape[:2], dtype=np.uint8)
for trg_color in trg_colors:
    current = cv2.inRange(mask, trg_color, trg_color)
    trg_mask = cv2.bitwise_or(trg_mask, current)
output_mask = mask.copy()
output_mask[trg_mask > 0] = (255, 255, 255)  # Set target pixels to white
# cv2.imwrite("test.png", output_mask)

In [ ]:
pixels = mask.reshape(-1, 3)
unique_colors = np.unique(pixels, axis=0)
print(unique_colors[:50])
print("總顏色數:", len(unique_colors))

In [ ]:
import cv2
import numpy as np

In [ ]:
mask = cv2.imread(
    "/mnt/c/Users/shh/Documents/ShunHsiangHsu/DefectSynthetic/rhino_modeling/rendered/runs/cube/20260318_020454/mask/view_003.png",
)
pixels = mask.reshape(-1, 3)
unique_colors = np.unique(pixels, axis=0)
print(unique_colors[:50])
print("總顏色數:", len(unique_colors))
np.unique(mask)

In [ ]:
render_root_dir = "/mnt/c/Users/shh/Documents/ShunHsiangHsu/DefectSynthetic/rhino_modeling/rendered/runs/cube/20260318_020454/"
depth_buffer_path = render_root_dir + "depth_buffer/view_004.pfm"
normal_buffer_path = render_root_dir + "normal_buffer/view_004.pfm"
depth_pfm = load_pfm(depth_buffer_path)
normal_pfm = load_pfm(normal_buffer_path)
print(np.unique(depth_pfm), depth_pfm.shape)
print(np.unique(normal_pfm), normal_pfm.shape)
depth_rendered_path = render_root_dir + "depth/view_004.png"
normal_rendered_path = render_root_dir + "normal/view_004.png"
depth_rendered = cv2.imread(depth_rendered_path, cv2.IMREAD_UNCHANGED)
normal_rendered = cv2.imread(normal_rendered_path, cv2.IMREAD_UNCHANGED)

In [ ]:
# depth: (H, W) float32
# normal: (H, W, 3) float32

fig, axes = plt.subplots(1, 3, figsize=(12, 4))


# 1) Visualize depth (ignore zeros as invalid/background)
d = depth_pfm.copy()
d[d <= 0] = np.nan

vmin = np.nanpercentile(d, 2)
vmax = np.nanpercentile(d, 98)
axes[0].imshow(d, cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("Depth")
axes[0].axis("off")
plt.colorbar(axes[0].imshow(d, cmap="viridis", vmin=vmin, vmax=vmax), ax=axes[0], label="Depth", shrink=0.5)

# Optional: inverse depth (near = bright)
inv = 1.0 / np.maximum(d, 1e-6)
inv[d <= 0] = np.nan
axes[1].imshow(inv, cmap="magma")
axes[1].set_title("Inverse Depth")
axes[1].axis("off")
plt.colorbar(axes[1].imshow(inv, cmap="magma"), ax=axes[1], label="1/depth", shrink=0.5)

# 2) Visualize normal as RGB ([-1,1] -> [0,1])
n = normal_pfm.copy()
n_norm = np.linalg.norm(n, axis=-1, keepdims=True)
n = n / np.maximum(n_norm, 1e-8)

n_rgb = (n + 1.0) * 0.5
n_rgb = np.clip(n_rgb, 0, 1)

# mask invalid where depth invalid
mask3d = (depth_pfm > 0).astype(np.float32)
if mask3d.ndim == 2:
    mask3d = mask3d[..., None]   # (H, W, 1)
n_rgb = n_rgb * mask3d

axes[2].imshow(n_rgb)
axes[2].set_title("Normal (RGB)")
axes[2].axis("off")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(depth_rendered, cmap="viridis")
axes[0].set_title("Depth (Rendered)")
axes[0].axis("off")
axes[1].imshow(normal_rendered)
axes[1].set_title("Normal (Rendered)")
axes[1].axis("off")

In [ ]:
color_mask = cv2.imread("/mnt/c/Users/shh/Documents/ShunHsiangHsu/DefectSynthetic/rhino_modeling/rendered/mask/view_004.png")

In [ ]:
green  = np.array([0, 255, 0], dtype=color_mask.dtype) # BGR
yellow = np.array([0, 255, 255], dtype=color_mask.dtype) # BGR
orange = np.array([0, 165, 255], dtype=color_mask.dtype) # BGR
white = np.array([255, 255, 255], dtype=color_mask.dtype) # BGR

green_mask  = (np.all(color_mask == green,  axis=-1).astype(np.uint8) * 255)
yellow_mask = (np.all(color_mask == yellow, axis=-1).astype(np.uint8) * 255)
orange_mask = (np.all(color_mask == orange, axis=-1).astype(np.uint8) * 255)
non_white_mask = (np.any(color_mask != white, axis=-1).astype(np.uint8) * 255)

for m, color_name in zip([green_mask, yellow_mask, orange_mask, non_white_mask], ["green", "yellow", "orange", "non_white"]):
    cv2.imwrite(f"{color_name}_mask.png", m)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(color_mask)
axes[1].imshow(green_mask, cmap="gray", vmin=0, vmax=255)
axes[2].imshow(yellow_mask, cmap="gray", vmin=0, vmax=255)
axes[3].imshow(orange_mask, cmap="gray", vmin=0, vmax=255)
for ax in axes: ax.axis("off")